# Create Datasets and Basic Summaries

In [1]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter
from datasets import load_dataset
from mapping import *
pd.set_option('display.max_columns', None)

/Users/briceshun/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_raw = 'data/raw'

In [3]:
def summary_corpus(
    data: pd.DataFrame,
    column: str
    ):
    # Get length of text
    df = data.copy()
    df["length"] = df[column].apply(lambda x: len(x))

    # Unique words in corpus
    dict_words = dict(Counter(" ".join(df[column].values.tolist()).split(" ")).items())
    corpus = list(dict_words.keys())

    print(f"Records: {len(df)}")
    print(f"Average length: {round(np.average(df['length']), 2)}")
    print(f"Vocabulary: {len(corpus)}")

    return corpus

## 1. Affective Text

In [4]:
data_affectivetext = load_dataset("sem_eval_2018_task_1", 'subtask5.english')
dic_affectivetext = data_affectivetext.data
df_affectivetext = pd.concat([
                    dic_affectivetext["train"].to_pandas().assign(dataset = 'train'),
                    dic_affectivetext["validation"].to_pandas().assign(dataset = 'validation'),
                    dic_affectivetext["test"].to_pandas().assign(dataset = 'test')
                ])
df_affectivetext.head()

,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust,dataset
0,2017-En-21441,“Worry is a down payment on a problem you may ...,False,True,False,False,False,False,True,False,False,False,True,train
1,2017-En-31535,Whatever you decide to do make sure it makes y...,False,False,False,False,True,True,True,False,False,False,False,train
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,True,False,True,False,True,False,True,False,False,False,False,train
3,2017-En-31436,Accept the challenges so that you can literall...,False,False,False,False,True,False,True,False,False,False,False,train
4,2017-En-22195,My roommate: it's okay that we can't spell bec...,True,False,True,False,False,False,False,False,False,False,False,train


In [5]:
corpus_affectivetext = summary_corpus(df_affectivetext, 'Tweet')

Records: 10983
Average length: 94.79
Vocabulary: 37378


In [6]:
# Standard format
df_affectivetext_processed = df_affectivetext.copy().reset_index(drop=True)
df_affectivetext_processed.rename(columns={'ID':'id', 'Tweet':'text'}, inplace=True)
df_affectivetext_processed['emotion'] = df_affectivetext_processed.apply(lambda x: [col for col in df_affectivetext_processed.columns if x[col] == True], axis=1)
for col in [i for i in df_affectivetext_processed.columns if i not in ['id', 'text', 'emotion']]:
    df_affectivetext_processed[col] = np.where(df_affectivetext_processed[col]==True, 1, 0)

df_affectivetext_processed.head()

,id,text,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust,dataset,emotion
0,2017-En-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1,0,"[anticipation, optimism, trust]"
1,2017-En-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0,0,"[joy, love, optimism]"
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0,0,"[anger, disgust, joy, optimism]"
3,2017-En-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0,0,"[joy, optimism]"
4,2017-En-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0,0,"[anger, disgust]"


In [7]:
print(f"Records: {len(df_affectivetext_processed)}")

Records: 10983


## 2. Crowdflower

In [8]:
path_crowdflower = "crowdflower"
df_crowdflower = pd.read_csv(f"{path_raw}/{path_crowdflower}/crowdflower.csv")
df_crowdflower = df_crowdflower[df_crowdflower['sentiment']!='empty']
df_crowdflower.head()

,tweet_id,sentiment,author,content
1,1956967666,sadness,wannamama,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,coolfunky,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,czareaquino,wants to hang out with friends SOON!
4,1956968416,neutral,xkilljoyx,@dannycastillo We want to trade with someone w...
5,1956968477,worry,xxxPEACHESxxx,Re-pinging @ghostridah14: why didn't you go to...


In [9]:
corpus_crowdflower = summary_corpus(df_crowdflower, 'content')

Records: 39173
Average length: 73.59
Vocabulary: 81981


In [10]:
df_crowdflower_processed = df_crowdflower.copy().reset_index(drop=True)

# Standard format
df_crowdflower_processed.rename(columns={'tweet_id':'id', 'content':'text', 'sentiment': 'emotion'}, inplace=True)
for col in df_crowdflower_processed['emotion'].unique():
    df_crowdflower_processed[col] = df_crowdflower_processed['emotion'].apply(lambda x: 1 if x == col else 0)

df_crowdflower_processed.head()

,id,emotion,author,text,sadness,enthusiasm,neutral,worry,surprise,love,fun,hate,happiness,boredom,relief,anger
0,1956967666,sadness,wannamama,Layin n bed with a headache ughhhh...waitin o...,1,0,0,0,0,0,0,0,0,0,0,0
1,1956967696,sadness,coolfunky,Funeral ceremony...gloomy friday...,1,0,0,0,0,0,0,0,0,0,0,0
2,1956967789,enthusiasm,czareaquino,wants to hang out with friends SOON!,0,1,0,0,0,0,0,0,0,0,0,0
3,1956968416,neutral,xkilljoyx,@dannycastillo We want to trade with someone w...,0,0,1,0,0,0,0,0,0,0,0,0
4,1956968477,worry,xxxPEACHESxxx,Re-pinging @ghostridah14: why didn't you go to...,0,0,0,1,0,0,0,0,0,0,0,0


In [11]:
print(f"Records: {len(df_crowdflower_processed)}")

Records: 39173


## 3. Electoral Tweets

In [12]:
path_electoraltweets = 'Annotated-US2012-Election-Tweets/Questionnaire'
# Questionnaire 1
df_electoraltweets1 = pd.read_csv(f"{path_raw}/{path_electoraltweets}1/AnnotatedTweets.txt", sep='	')
# Questionnaire 2
df_electoraltweets2_1 = pd.read_csv(f"{path_raw}/{path_electoraltweets}2/Batch1/AnnotatedTweets.txt", sep='	')
df_electoraltweets2_2 = pd.read_csv(f"{path_raw}/{path_electoraltweets}2/Batch2/AnnotatedTweets.txt", sep='	')
# Merge
df_electoraltweets = pd.concat([df_electoraltweets2_1, df_electoraltweets2_2])
df_electoraltweets = df_electoraltweets[df_electoraltweets['q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion']!='BLANK']
df_electoraltweets.head()

,unitid,createdat,golden,id,missed,startedat,tainted,channel,trust,workerid,country,region,city,tweet,q1whoisfeelingorwhofeltanemotioninotherwordswhoisthesourceoftheemotion,q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion,fontcolorolivetweetertweetfontbrq3ifthereisabetterwordfordescribingtheemotionthantheoneslistedabovethentypeithere,q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq4thenpleasetellusiftheemotioninthistweetispositivenegativeorneither,fontcolorolivetweetertweetfontbrq5howstronglyistheemotionbeingexpressedinthistweet,q6towardswhomorwhatinotherwordswhoorwhatisthetargetoftheemotion,fontcolorolivetweetertweetfontbrq7whichwordsinthetweethelpidentifyingtheemotion,q8whatreasoncanbededucedfromthetweetfortheemotionwhatisthecauseoftheemotion,fontcolorolivetweetertweetfontbrq9thistweetisaboutwhichofthefollowingissues,fontcolorolivetweetertweetfontbrq10ifthetweetisaboutanissuenotlistedabovethentypeithere,q11whichofthefollowingbestdescribesthepurposeofthistweet,origgolden,origq11whichofthefollowingbestdescribesthepurposeofthistweet,q11whichofthefollowingbestdescribesthepurposeofthistweetgold,q11whichofthefollowingbestdescribesthepurposeofthistweetgoldreason,fontcolorgreycommentsfont,q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq3thenpleasetellusiftheemotioninthistweetispositivenegativeorneither,q6bwhichofthesebestdescribesthetargetoftheemotion
0,234318245,12/17/2012 183705,True,775142387,BLANK,12/17/2012 181543,False,amt,1.0,11338794,USA,GA,Covington,"I'm a #Republican, but if I have to hear my mo...",Tweeter,disgust,Sarcasm,negative emotion,the emotion is being expressed with medium int...,my mom,I'm going to stab myself with my bayonet.,I have to hear my mom talk about #Romney,"About the election process, election publicity...",BLANK,"to point out hypocrisy, to disagree, to ridicu...",1,BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,NaN,NaN,NaN
1,234318378,12/16/2012 025145,False,772925098,BLANK,12/16/2012 022248,False,amt,1.0,11338794,USA,GA,Covington,Will Obama fire the person responsible for thi...,Tweeter,anger or annoyance or hostility or fury,BLANK,negative emotion,the emotion is being expressed with a high int...,not specified,"fire misguided individuals, hurt",hurt,None of the above,BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN
2,234318670,12/17/2012 185740,False,775168524,BLANK,12/17/2012 183815,False,amt,1.0,11338794,USA,GA,Covington,haha @DickMorrisTweet Romney is going to have ...,Tweeter,anger or annoyance or hostility or fury,belittle,neither positive nor negative,the emotion is being expressed with medium int...,DickMorrisTweet,haha,Romney is going to have a great convention. It...,"About the election process, election publicity...",BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN
3,234318253,12/17/2012 183705,False,775142384,BLANK,12/17/2012 181543,False,amt,1.0,11338794,USA,GA,Covington,S/0 to my newest @freeboosieRS &amp vote for O...,Tweeter,anticipation or expectancy or interest,BLANK,BLANK,the emotion is being expressed with medium int...,my newest @freeboosieRS,L0L,vote for Obama & boosie will soon to be free,About Social and Civil Issues but not related ...,BLANK,"to motivate, to entertain, or to provide infor...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN
4,234318724,12/17/2012 185740,False,775168523,BLANK,12/17/2012 183815,False,amt,1.0,11338794,USA,GA,Covington,Nicki Minaj Fucked Up With That Mitt Romney Li...,Tweeter,anger or annoyance or hostility or fury,BLANK,negative emotion,the emotion is being expressed with a high int...,Nicki Minaj,Fucked Up,Mitt Romney Line,"About the election process, election publicity...",BLANK,"to point out hypocrisy, to disagree, to ridicu...",BLANK,BLANK,BLANK,BLANK,NaN,NaN,NaN


In [13]:
corpus_electoraltweets = summary_corpus(df_electoraltweets, 'tweet')

Records: 4055
Average length: 102.56
Vocabulary: 6695


In [14]:
df_electoraltweets_processed = df_electoraltweets.copy().reset_index(drop=True)

# Standard format
keep_cols = ['id', 'tweet',
            'q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion',
            'q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq4thenpleasetellusiftheemotioninthistweetispositivenegativeorneither',
            'fontcolorolivetweetertweetfontbrq5howstronglyistheemotionbeingexpressedinthistweet'
            ]
df_electoraltweets_processed = df_electoraltweets_processed[keep_cols]
df_electoraltweets_processed\
    .rename(columns={   'q2whatemotionchooseoneoftheoptionsfrombelowthatbestrepresentstheemotion':'emotion',
                        'q4ifwhenansweringq2youhavechosenanemotionfromtheotheremotionscategoryorifyouansweredq4thenpleasetellusiftheemotioninthistweetispositivenegativeorneither': 'sentiment',
                        'fontcolorolivetweetertweetfontbrq5howstronglyistheemotionbeingexpressedinthistweet': 'intensity'
                    },
            inplace=True
            )
df_electoraltweets_processed['sentiment'] = df_electoraltweets_processed['sentiment'].apply(lambda x: re.sub(r' emotion| positive nor negative', '', x) if type(x)==str else 'BLANK')
df_electoraltweets_processed['intensity'] = df_electoraltweets_processed['intensity'].apply(lambda x: re.sub(r'.* with a |.* with | intensity', '', x))

# Pivot emotions
df_electoraltweets_processed['emotion'] = df_electoraltweets_processed['emotion'].apply(lambda x: re.sub(r'\s{2,}', ' ', x))
df_electoraltweets_processed['emotion_'] = df_electoraltweets_processed['emotion'].apply(lambda x: re.sub(r'\s+', '_', x))
for col in df_electoraltweets_processed['emotion_'].unique():
    df_electoraltweets_processed[col] = df_electoraltweets_processed['emotion_'].apply(lambda x: 1 if x == col else 0)
df_electoraltweets_processed['emotion'] = df_electoraltweets_processed.apply(lambda x: [col for col in df_electoraltweets_processed.columns if x[col] == True], axis=1)

df_electoraltweets_processed.drop(columns=['emotion_'], inplace=True)
df_electoraltweets_processed.rename(columns={'tweet':'text'}, inplace=True)

df_electoraltweets_processed.head()

,id,text,emotion,sentiment,intensity,disgust,anger_or_annoyance_or_hostility_or_fury,anticipation_or_expectancy_or_interest,admiration,dislike,uncertainty_or_indecision_or_confusion,joy_or_happiness_or_elation,like,indifference,disappointment,calmness_or_serenity,hate,amazement,acceptance,fear_or_apprehension_or_panic_or_terror,vigilance,trust,surprise,sadness_or_gloominess_or_grief_or_sorrow
0,775142387,"I'm a #Republican, but if I have to hear my mo...",[disgust],negative,medium,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,772925098,Will Obama fire the person responsible for thi...,[anger_or_annoyance_or_hostility_or_fury],negative,high,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,775168524,haha @DickMorrisTweet Romney is going to have ...,[anger_or_annoyance_or_hostility_or_fury],neither,medium,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,775142384,S/0 to my newest @freeboosieRS &amp vote for O...,[anticipation_or_expectancy_or_interest],BLANK,medium,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,775168523,Nicki Minaj Fucked Up With That Mitt Romney Li...,[anger_or_annoyance_or_hostility_or_fury],negative,high,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [15]:
print(f"Records: {len(df_electoraltweets_processed)}")

Records: 4055


# 4. EmoInt

In [16]:
path_emoint_dev = 'EmoInt/EmoInt Dev'
path_emoint_test = 'EmoInt/EmoInt Test Gold'
path_emoint_train = 'EmoInt/EmoInt Train'
# Load all
l_emotions = ['anger', 'fear', 'joy', 'sadness']
l_emoint = []
for path_emoint, dataset in zip([path_emoint_dev, path_emoint_test, path_emoint_train], ['dev.target', 'test.gold', 'train']):
    for emotion in l_emotions:
        df = pd.read_table(f"{path_raw}/{path_emoint} Data_{emotion}-ratings-0to1.{dataset}.txt", 
                            delimiter='	',
                            names=['id', 'tweet', 'emotion', 'NONE'])
        df['dataset'] = dataset
        l_emoint.append(df)
# Merge
df_emoint = pd.concat(l_emoint).drop(columns=['NONE'])
df_emoint.head()

,id,tweet,emotion,dataset
0,10857,@ZubairSabirPTI pls dont insult the word 'Molna',anger,dev.target
1,10858,@ArcticFantasy I would have almost took offens...,anger,dev.target
2,10859,@IllinoisLoyalty that Rutgers game was an abom...,anger,dev.target
3,10860,@CozanGaming that's what lisa asked before she...,anger,dev.target
4,10861,Sometimes I get mad over something so minuscul...,anger,dev.target


In [17]:
corpus_emoint = summary_corpus(df_emoint, 'tweet')

Records: 7102
Average length: 95.25
Vocabulary: 24535


In [18]:
# Standard Format
df_emoint_processed = df_emoint.copy().reset_index(drop=True)
df_emoint_processed.rename({'tweet': 'text'}, inplace=True)
for col in df_emoint_processed['emotion'].unique():
    df_emoint_processed[col] = df_emoint_processed['emotion'].apply(lambda x: 1 if x == col else 0)
df_emoint_processed['emotion'] = df_emoint_processed.apply(lambda x: [col for col in df_emoint_processed.columns if x[col] == True], axis=1)
df_emoint_processed['disgust'], df_emoint_processed['surprise'] = 0, 0
df_emoint_processed.loc[df_emoint_processed['dataset'] == 'dev.target', 'dataset'] = 'validation'
df_emoint_processed.loc[df_emoint_processed['dataset'] == 'test.gold', 'dataset'] = 'test'

df_emoint_processed.rename(columns={'tweet':'text'}, inplace=True)

df_emoint_processed.head()

,id,text,emotion,dataset,anger,fear,joy,sadness,disgust,surprise
0,10857,@ZubairSabirPTI pls dont insult the word 'Molna',[anger],validation,1,0,0,0,0,0
1,10858,@ArcticFantasy I would have almost took offens...,[anger],validation,1,0,0,0,0,0
2,10859,@IllinoisLoyalty that Rutgers game was an abom...,[anger],validation,1,0,0,0,0,0
3,10860,@CozanGaming that's what lisa asked before she...,[anger],validation,1,0,0,0,0,0
4,10861,Sometimes I get mad over something so minuscul...,[anger],validation,1,0,0,0,0,0


In [19]:
print(f"Records: {len(df_emoint_processed)}")

Records: 7102


# 5. Go Emotions

In [20]:
path_goemotions = 'goemotions'
# Load
data_goemotions = load_dataset("go_emotions")
dic_goemotions = data_goemotions.data
df_goemotions = pd.concat([
                    dic_goemotions["train"].to_pandas().assign(dataset = 'train'),
                    dic_goemotions["validation"].to_pandas().assign(dataset = 'validation'),
                    dic_goemotions["test"].to_pandas().assign(dataset = 'test')
                ])

In [21]:
corpus_goemotions = summary_corpus(df_goemotions, 'text')

Records: 54263
Average length: 68.33
Vocabulary: 65524


In [22]:
# Load Mappings
with open(f"{path_raw}/{path_goemotions}/emotions.txt", "r") as file:
    taxonomy = file.read().split("\n")

with open(f"{path_raw}/{path_goemotions}/ekman_mapping.json", "r") as file:
    mapping = json.load(file)

mapping |= {'neutral': ['neutral']}
taxonomy_ekman = list(mapping.keys())

# Mapping function
def labelEncoding(
    input: pd.DataFrame
    ) -> pd.DataFrame:

    df = input

    # Defining a function that maps each index to emotion labels
    def idx2class(idx_list):
        arr = []
        for i in idx_list:
            taxonomy_value = taxonomy[int(i)]
            ekman_value = [k for k, v in mapping.items() if taxonomy_value in v][0]
            arr.append(ekman_value)
        return arr
    
    def idx2class_full(idx_list):
        arr_full = []
        for i in idx_list:
            taxonomy_value = taxonomy[int(i)]
            arr_full.append(taxonomy_value)
        return arr_full

    # Applying the function
    df['emotions_full'] = df['labels'].apply(idx2class_full)
    df['emotions'] = df['labels'].apply(idx2class)

    # OneHot encoding for multi-label classification
    for emo in taxonomy:
        df[emo] = np.zeros((len(df),1))
        df[emo] = df['emotions_full'].apply(lambda x: 1 if emo in x else 0)
    for emo in taxonomy_ekman:
        df['primary_' + emo] = np.zeros((len(df),1))
        df['primary_' + emo] = df['emotions'].apply(lambda x: 1 if emo in x else 0)
    for emo in taxonomy_ekman:
        df['secondary_' + emo] = np.zeros((len(df),1))
        df['secondary_' + emo] = df['emotions'].apply(lambda x: 1 if emo in x else 0)

    # Keeping only necessary columns
    df = df.drop(columns=['labels', 'emotions_full', 'emotions'], axis=1)

    return df

# Encode
df_goemotions = labelEncoding(df_goemotions).reset_index(drop=True)
df_goemotions.head()

,text,id,dataset,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,disappointment,disapproval,disgust,embarrassment,excitement,fear,gratitude,grief,joy,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral,primary_anger,primary_disgust,primary_fear,primary_joy,primary_sadness,primary_surprise,primary_neutral,secondary_anger,secondary_disgust,secondary_fear,secondary_joy,secondary_sadness,secondary_surprise,secondary_neutral
0,My favourite food is anything I didn't have to...,eebbqej,train,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1
1,"Now if he does off himself, everyone will thin...",ed00q6i,train,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1
2,WHY THE FUCK IS BAYLESS ISOING,eezlygj,train,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
3,To make her feel threatened,ed7ypvh,train,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,Dirty Southern Wankers,ed0bdzj,train,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0


In [23]:
print(f"Records: {len(df_goemotions)}")

Records: 54263


## 6. SSEC

In [24]:
path_ssec = 'ssec/ssec-aggregated'
threshold = '0.99'
df_ssec = pd.concat([
            pd.read_csv(f"{path_raw}/{path_ssec}/test-combined-{threshold}.csv", 
                        sep='\t', 
                        names=['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'tweet'])\
                .assign(dataset='test'),
            pd.read_csv(f"{path_raw}/{path_ssec}/train-combined-{threshold}.csv", 
                        sep='\t', 
                        names=['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'tweet'])\
                .assign(dataset='train'),
            ])
df_ssec['tweet'] = df_ssec['tweet'].astype(str)
df_ssec.head()

,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,tweet,dataset
0,---,---,---,---,---,---,---,---,He who exalts himself shall be humbled; a...,test
1,---,---,---,---,---,---,---,Trust,RT @prayerbullets: I remove Nehushtan -previou...,test
2,---,---,---,---,---,---,---,Trust,@Brainman365 @heidtjj @BenjaminLives I have so...,test
3,---,---,---,---,---,---,---,---,#God is utterly powerless without Human interv...,test
4,---,---,---,---,---,---,---,---,@David_Cameron Miracles of #Multiculturalism...,test


In [25]:
corpus_ssec = summary_corpus(df_ssec, 'tweet')

Records: 4870
Average length: 108.18
Vocabulary: 21082


In [26]:
df_ssec_processed = df_ssec.copy().reset_index(drop=True)
df_ssec_processed.reset_index(inplace=True)
for col in [i for i in df_ssec_processed.columns if i not in ['index', 'tweet', 'dataset']]:
    df_ssec_processed[col] = df_ssec_processed[col].apply(lambda x: 1 if x != '---' else 0)
df_ssec_processed['emotion'] = df_ssec_processed.apply(lambda x: [col for col in df_ssec_processed.columns if x[col] == True], axis=1)
df_ssec_processed['noemotion'] = np.where(
                                    (df_ssec_processed['anger'] == False) &
                                    (df_ssec_processed['anticipation'] == False) &
                                    (df_ssec_processed['disgust'] == False) &
                                    (df_ssec_processed['fear'] == False) &
                                    (df_ssec_processed['joy'] == False) &
                                    (df_ssec_processed['sadness'] == False) &
                                    (df_ssec_processed['surprise'] == False) &
                                    (df_ssec_processed['trust'] == False),
                                    1, 0)

df_ssec_processed.rename(columns={'index': 'id', 'tweet': 'text'}, inplace=True)
df_ssec_processed = df_ssec_processed.loc[df_ssec_processed['noemotion'] == 0, df_ssec_processed.columns[:-1]]
df_ssec_processed = df_ssec_processed[['id', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'text', 'dataset']]

df_ssec_processed.head()

,id,anger,anticipation,disgust,fear,joy,sadness,surprise,trust,text,dataset
1,1,0,0,0,0,0,0,0,1,RT @prayerbullets: I remove Nehushtan -previou...,test
2,2,0,0,0,0,0,0,0,1,@Brainman365 @heidtjj @BenjaminLives I have so...,test
5,5,0,0,0,0,1,0,0,1,This world needs a tight group hug. Tight enou...,test
7,7,0,0,0,0,0,0,0,1,A Godly husband - knows you - trusts you - lo...,test
9,9,1,0,0,0,0,0,0,0,#BIBLE = Big Irrelevant Book of Lies and Exagg...,test


In [27]:
print(f"Records: {len(df_ssec_processed)}")

Records: 1534


## 7. TEC

In [28]:
# Load Twitter Emotion Corpus
TEC_url='https://archive.org/download/misc-dataset/TEC.csv'
df_TEC=pd.read_csv(TEC_url)
df_TEC.head()

,tid,text,emotion
0,145353048817012736,Thinks that @melbahughes had a great 50th birt...,surprise
1,144279638024257536,"Como una expresión tan simple, una sola oració...",sadness
2,140499585285111809,the moment when you get another follower and y...,joy
3,145207578270507009,Be the greatest dancer of your life! practice ...,joy
4,139502146390470656,eww.. my moms starting to make her annual rum ...,disgust


In [29]:
corpus_TEC = summary_corpus(df_TEC, 'text')

Records: 21051
Average length: 84.71
Vocabulary: 55501


In [30]:
# Standard Format
df_TEC_processed = df_TEC.copy().reset_index(drop=True)
df_TEC_processed.rename(columns={'tid': 'id'}, inplace=True)
for col in df_TEC_processed['emotion'].unique():
    df_TEC_processed[col] = df_TEC_processed['emotion'].apply(lambda x: 1 if x == col else 0)
df_TEC_processed['emotion'] = df_TEC_processed.apply(lambda x: [col for col in df_TEC_processed.columns if x[col] == True], axis=1)

df_TEC_processed.head()

,id,text,emotion,surprise,sadness,joy,disgust,fear,anger
0,145353048817012736,Thinks that @melbahughes had a great 50th birt...,[surprise],1,0,0,0,0,0
1,144279638024257536,"Como una expresión tan simple, una sola oració...",[sadness],0,1,0,0,0,0
2,140499585285111809,the moment when you get another follower and y...,[joy],0,0,1,0,0,0
3,145207578270507009,Be the greatest dancer of your life! practice ...,[joy],0,0,1,0,0,0
4,139502146390470656,eww.. my moms starting to make her annual rum ...,[disgust],0,0,0,1,0,0


In [31]:
print(f"Records: {len(df_TEC_processed)}")

Records: 21051


## Convert to Ekman

In [32]:
def toEkman(
    data: pd.DataFrame,
    mapping: dict
    ) -> pd.DataFrame:

    df = data.copy()

    for emotion in mapping.keys():
        for emotype in mapping[emotion].keys():
            if mapping[emotion][emotype] != []:
                df[emotype + '_' + emotion] = np.where((df[mapping[emotion][emotype]]==1).any(axis=1), 1, 0)
            else:
                df[emotype + '_' + emotion] = 0

    return df

In [33]:
# Apply mapping
df_affectivetext_ekman = toEkman(df_affectivetext_processed, map_affectivetext)
df_crowdflower_ekman = toEkman(df_crowdflower_processed, map_crowdflower)
df_electoraltweets_ekman = toEkman(df_electoraltweets_processed, map_electoraltweets)
df_ssec_ekman = toEkman(df_ssec_processed, map_ssec)

df_emoint_ekman = df_emoint_processed.copy()
df_goemotions_ekman = df_goemotions.copy()
df_TEC_ekman = df_TEC_processed.copy()

for col in map_ekman:
    df_emoint_ekman['primary_' + col] = df_emoint_ekman[col]
    df_TEC_ekman['primary_' + col]  = df_TEC_ekman[col]

    df_emoint_ekman['secondary_' + col] = df_emoint_ekman[col]
    df_TEC_ekman['secondary_' + col]  = df_TEC_ekman[col]

In [34]:
# Rearrange
l_process = zip(
    [df_affectivetext_ekman, df_crowdflower_ekman, df_electoraltweets_ekman, df_ssec_ekman, df_emoint_ekman, df_goemotions, df_TEC_ekman],
    ['affectivetext', 'crowdflower', 'electoraltweets', 'ssec', 'emoint', 'goemotions', 'tec']
)
l_full = []
l_primary = []
l_secondary = []

for df, name in l_process:
    df = df.assign(source = name)
    if 'dataset' not in df.columns:
        df['dataset'] = 'all'

    base_cols = ['source', 'dataset', 'id', 'text']

    col_full = base_cols + map_all[name]
    col_primary = base_cols + ['primary_' + i for i in map_ekman]
    col_secondary = base_cols + ['secondary_' + i for i in map_ekman]

    # Remove empty rows
    df_full = df[col_full].copy()
    df_full = df_full.assign(noemo = df_full[map_all[name]].sum(axis=1))
    df_full = df_full[df_full['noemo']!=0].drop(columns=['noemo'])

    df_primary = df[col_primary].copy()
    df_primary.rename(columns={'primary_' + i:i for i in map_ekman}, inplace=True)
    df_primary = df_primary.assign(noemo = df_primary[map_ekman].sum(axis=1))
    df_primary = df_primary[df_primary['noemo']!=0].drop(columns=['noemo'])

    df_secondary = df[col_secondary].copy()
    df_secondary.rename(columns={'secondary_' + i:i for i in map_ekman}, inplace=True)
    df_secondary = df_secondary.assign(noemo = df_secondary[map_ekman].sum(axis=1))
    df_secondary = df_secondary[df_secondary['noemo']!=0].drop(columns=['noemo'])

    print(f"{name}")
    print(f"|-- Raw={len(df)} | Original={len(df_full)} | Primary={len(df_primary)} | Secondary={len(df_secondary)}")

    l_full.append(df_full)
    l_primary.append(df_primary)
    l_secondary.append(df_secondary)

affectivetext
|-- Raw=10983 | Original=10690 | Primary=10248 | Secondary=10690
crowdflower
|-- Raw=39173 | Original=39173 | Primary=12671 | Secondary=30535
electoraltweets
|-- Raw=4055 | Original=4055 | Primary=1362 | Secondary=3935
ssec
|-- Raw=1534 | Original=1534 | Primary=1193 | Secondary=1534
emoint
|-- Raw=7102 | Original=7102 | Primary=7102 | Secondary=7102
goemotions
|-- Raw=54263 | Original=54263 | Primary=38242 | Secondary=38242
tec
|-- Raw=21051 | Original=21051 | Primary=21051 | Secondary=21051


### Export

In [35]:
# Separate
for data_list, data_variant in zip([l_full, l_primary, l_secondary], ['original', 'primary', 'secondary']):
    for data, name in zip(data_list, ['affectivetext', 'crowdflower', 'electoraltweets', 'ssec', 'emoint', 'goemotions', 'tec']):
        data.to_parquet(f'data//final//{data_variant}//{name}.parquet', index=False)